# Amazon Neptune 图存储

## 概述

本 Notebook 介绍 Semantica 中对 Amazon Neptune 数据库的集成。Amazon Neptune 是一项完全托管的图数据库服务,同时支持属性图(通过 OpenCypher/Gremlin)和 RDF 图(通过 SPARQL)。

### 关键特性

- **IAM 认证**:通过 AuthManager 使用 AWS SigV4 签名进行安全访问
- **OpenCypher 支持**:使用标准 OpenCypher 语法进行查询
- **Bolt 协议**:使用 Neo4j Bolt 驱动实现高效的二进制通信
- **原生 ~id 支持**:利用 Neptune 原生的元素 ID 处理机制
- **完整的 CRUD 操作**:创建、读取、更新、删除节点和关系
- **自动重试**:内置带指数退避的重试逻辑,处理瞬时错误

### 前提条件

- 一个 Amazon Neptune 数据库集群
- 已配置 AWS 凭据(boto3、环境变量或 IAM 角色)
- 能够通过网络访问你的 Neptune 集群(VPC、安全组)
- 你的公网 IP 地址或 VPN/办公网络 CIDR(运行 `curl ifconfig.me` 可查到你的公网 IP),下文将用它来限制对数据库的访问

#### 使用 CloudFormation 快速搭建

如果你没有 Neptune 集群,可以使用提供的 CloudFormation 模板来创建一个带公网端点和 IAM 认证的集群:

```bash
# 部署 Neptune 堆栈(大约需要 15-20 分钟)
# 将 203.0.113.25/32 替换为你自己的公网 IP(运行 `curl ifconfig.me` 查询)
# 或你的办公室/VPN CIDR。这会在网络层面限制谁可以访问数据库
# —— 除了短暂的本地实验之外,切勿将其放宽到 0.0.0.0/0。
aws cloudformation create-stack \
  --stack-name semantica-neptune \
  --template-body file://neptune-setup.yaml \
  --parameters ParameterKey=ClientCidr,ParameterValue=203.0.113.25/32 \
  --capabilities CAPABILITY_NAMED_IAM

# 等待堆栈创建完成
aws cloudformation wait stack-create-complete --stack-name semantica-neptune

# 获取输出(端点、端口、凭据)
aws cloudformation describe-stacks --stack-name semantica-neptune \
  --query 'Stacks[0].Outputs' --output table
```

该模板会创建:
- 带公有子网、Internet Gateway 和 VPC Flow Logs(输出到 CloudWatch Logs)的 VPC
- 启用了 IAM 认证的 Neptune 集群(`db.t3.medium`)
- 针对 OpenCypher 查询拥有最小权限访问的 IAM 用户
- 仅允许来自你所指定的 `ClientCidr` 访问 Bolt 协议(端口 8182)的安全组

> ⚠️ **安全提示**:为了在演示/测试环境中保持简单,该模板创建了一个带静态访问密钥的 IAM 用户。在生产环境中,我们建议使用 IAM 角色(EC2 实例角色、ECS 任务角色、Lambda 执行角色),它们会提供自动轮换的临时凭据。CloudFormation 输出中的 secret access key 是明文提供的,目的是简化初始配置 —— 在生产环境中请使用 AWS Secrets Manager。`ClientCidr` 参数是必填项(没有默认值),正是为了确保数据库不会被悄悄暴露给整个互联网。

**输出:**
- `NeptuneEndpoint` - 集群主机名(用作 `NEPTUNE_ENDPOINT`)
- `NeptunePort` - 8182(用作 `NEPTUNE_PORT`)
- `AwsAccessKeyId` - IAM 用户访问密钥(用作 `AWS_ACCESS_KEY_ID`)
- `AwsSecretAccessKey` - 以**明文**形式给出的 IAM 用户私有密钥(用作 `AWS_SECRET_ACCESS_KEY`)
- `AwsRegion` - 部署区域(用作 `AWS_REGION`)

**清理:**
```bash
aws cloudformation delete-stack --stack-name semantica-neptune
```

**预估月度成本(在 100% 利用率下约 100-105 美元/月):**

| 资源 | 成本(美元) |
| --- | --- |
| Neptune db.t3.medium 实例 | ~96/月(0.132/小时) |
| 存储(10 GB) | ~1/月 |
| I/O 请求 | ~1-5/月 |
| 公网 IPv4 地址 | ~3.60/月(0.005/小时) |
| VPC Flow Logs(CloudWatch Logs) | ~1-2/月,视流量而定 |
| VPC、子网、路由表、Internet Gateway、IAM | 无额外费用 |

> **免费额度**:Neptune 新用户可获得 30 天免费(750 小时的 db.t3.medium、1000 万次 I/O、1 GB 存储)。不使用时请删除堆栈以避免产生费用。

---

## 安装

```bash
# 安装带 Neptune 支持的 Semantica
pip install semantica

# 必需的依赖(自动安装)
pip install boto3 neo4j
```

In [ ]:
!pip install semantica

## 配置

设置你的 Neptune 集群端点和 AWS 凭据。请将占位符替换为你的实际配置。

In [ ]:
import os

# Neptune cluster configuration - REPLACE WITH YOUR VALUES
# (Get these from CloudFormation stack outputs)
os.environ["NEPTUNE_ENDPOINT"] = "your-cluster.us-east-1.neptune.amazonaws.com"
os.environ["NEPTUNE_PORT"] = "8182"
os.environ["AWS_REGION"] = "us-east-1"

# AWS credentials for IAM Authentication
# Option 1: IAM User (static credentials from CloudFormation template)
# os.environ["AWS_ACCESS_KEY_ID"] = "AKIA..."  # From AwsAccessKeyId output
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."  # From AwsSecretAccessKey output
# Note: No AWS_SESSION_TOKEN needed for IAM users

# Option 2: IAM Role / Temporary credentials (e.g., STS AssumeRole, EC2 instance role)
# os.environ["AWS_ACCESS_KEY_ID"] = "ASIA..."  # Temporary access key
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."  # Temporary secret key
# os.environ["AWS_SESSION_TOKEN"] = "..."      # REQUIRED for temporary credentials

print(f"Neptune Endpoint: {os.environ.get('NEPTUNE_ENDPOINT')}")
print(f"AWS Region: {os.environ.get('AWS_REGION')}")

## 步骤 1:初始化 Neptune 存储

使用 IAM 认证初始化到你的 Amazon Neptune 集群的连接。

In [ ]:
import os
from semantica.graph_store import GraphStore

# Option 1: Using GraphStore factory (recommended)
neptune_store = GraphStore(
    backend="neptune",
    endpoint=os.environ.get("NEPTUNE_ENDPOINT"),
    port=int(os.environ.get("NEPTUNE_PORT", 8182)),
    region=os.environ.get("AWS_REGION", "us-east-1"),
    iam_auth=True,
)

# Connect to Neptune
neptune_store.connect()
print("Connected to Amazon Neptune!")

### 无 IAM 认证的开发/测试

对于不需要 IAM 认证的开发或测试环境(例如 Neptune notebooks 或仅限 VPC 访问),你可以禁用 IAM 签名:

In [ ]:
# For dev/test environments without IAM authentication
neptune_store_dev = GraphStore(
    backend="neptune",
    endpoint=os.environ.get("NEPTUNE_ENDPOINT"),
    port=int(os.environ.get("NEPTUNE_PORT", 8182)),
    region=os.environ.get("AWS_REGION", "us-east-1"),
    iam_auth=False,  # Disable IAM signing for dev/test
)
neptune_store_dev.connect()

### 认证选项

IAM 认证(推荐用于生产环境)会自动使用 AWS 凭据链:
1. 环境变量(AWS_ACCESS_KEY_ID、AWS_SECRET_ACCESS_KEY)
2. AWS 凭据文件(~/.aws/credentials)
3. IAM 角色(适用于 EC2、Lambda、ECS)

## 步骤 2:节点操作

### 创建节点

节点代表图中的实体。每个节点可以拥有:
- **ID**:唯一标识符(自定义或自动生成的 UUID)
- **标签(Labels)**:类别/类型(例如 `Person`、`Company`)
- **属性(Properties)**:键值对(例如 `{"name": "Alice", "age": 30}`)

In [ ]:
# Create a single node with custom ID (id in properties)
alice = neptune_store.create_node(
    labels=["Person"],
    properties={"id": "alice", "name": "Alice", "age": 30, "role": "Engineer"}
)
print(f"Created node: {alice}")

# Create a node with auto-generated UUID (no id in properties)
bob = neptune_store.create_node(
    labels=["Person"],
    properties={"name": "Bob", "age": 25, "role": "Designer"}
)
print(f"Created node with UUID: {bob['id']}")

# Create a company node with auto-generated ID
acme = neptune_store.create_node(
    labels=["Company"],
    properties={"name": "Acme Corp", "industry": "Technology", "founded": 2010}
)
print(f"Created company: {acme}")

### 创建多个节点(批量)

In [ ]:
# Batch create nodes for better performance
# Include 'id' in properties for custom IDs
nodes_data = [
    {"labels": ["Person"], "properties": {"id": "charlie", "name": "Charlie", "age": 35}},
    {"labels": ["Person"], "properties": {"id": "diana", "name": "Diana", "age": 28}},
    {"labels": ["Location"], "properties": {"name": "San Francisco", "state": "CA"}},
]

created_nodes = neptune_store.create_nodes(nodes_data)
print(f"Created {len(created_nodes)} nodes in batch")

### 检索节点

In [ ]:
# Get a specific node by ID
alice_node = neptune_store.get_node(node_id="alice")
print(f"Retrieved: {alice_node}")

# Get nodes by label
people = neptune_store.get_nodes(labels=["Person"], limit=10)
print(f"Found {len(people)} Person nodes:")
for person in people:
    print(f"  - {person.get('properties', {}).get('name')}")

# Get nodes by properties
engineers = neptune_store.get_nodes(
    labels=["Person"],
    properties={"role": "Engineer"},
    limit=5
)
print(f"Found {len(engineers)} engineers")

### 更新节点

In [ ]:
# Update node properties (merge mode - default)
updated_alice = neptune_store.update_node(
    node_id="alice",
    properties={"age": 31, "department": "AI Research"},
    merge=True
)
print(f"Updated Alice: {updated_alice}")

# Replace all properties (merge=False)
# WARNING: This removes properties not in the update
replaced = neptune_store.update_node(
    node_id="charlie",
    properties={"name": "Charlie", "age": 36},
    merge=False
)

### 删除节点

In [ ]:
# Delete a node (with detach=True to also delete relationships)
deleted = neptune_store.delete_node(node_id="diana", detach=True)
print(f"Deleted diana: {deleted}")

# Without detach (fails if node has relationships)
# neptune_store.delete_node(node_id="alice", detach=False)

## 步骤 3:关系操作

### 创建关系

关系连接节点,表示实体之间的联系。

In [ ]:
# Create a relationship between Alice and Acme
works_at = neptune_store.create_relationship(
    start_node_id="alice",
    end_node_id=acme["id"],
    rel_type="WORKS_AT",
    properties={"since": 2020, "position": "Senior Engineer"}
)
print(f"Created relationship: {works_at}")

# Create a KNOWS relationship between people
knows_rel = neptune_store.create_relationship(
    start_node_id="alice",
    end_node_id=bob["id"],
    rel_type="KNOWS",
    properties={"since": 2019}
)

### 检索关系

In [ ]:
# Get all relationships for a node
alice_rels = neptune_store.get_relationships(node_id="alice", direction="both")
print(f"Alice has {len(alice_rels)} relationships")

# Get outgoing relationships only
outgoing = neptune_store.get_relationships(node_id="alice", direction="out")

# Filter by relationship type
works_rels = neptune_store.get_relationships(
    node_id="alice",
    rel_type="WORKS_AT",
    direction="out"
)
print(f"Alice's work relationships: {len(works_rels)}")

### 删除关系

In [ ]:
# Delete a specific relationship by ID
if works_at.get("id"):
    deleted = neptune_store.delete_relationship(rel_id=works_at["id"])
    print(f"Deleted relationship: {deleted}")

## 步骤 4:OpenCypher 查询

Amazon Neptune 通过 Bolt 协议支持 OpenCypher 查询。使用标准的 Cypher 语法执行复杂的图模式。

In [ ]:
# Simple query
results = neptune_store.execute_query(
    "MATCH (p:Person) RETURN p.name, p.age ORDER BY p.age"
)
print("People in the graph:")
for record in results.get("records", []):
    print(f"  - {record.get('p.name')}: {record.get('p.age')} years old")

In [ ]:
# Using parameters (safer and more efficient)
results = neptune_store.execute_query(
    "MATCH (p:Person) WHERE p.age > $min_age RETURN p.name, p.age",
    parameters={"min_age": 25}
)
print(f"People over 25: {len(results.get('records', []))}")

In [ ]:
# Find relationships between nodes
results = neptune_store.execute_query("""
    MATCH (p:Person)-[r:WORKS_AT]->(c:Company)
    RETURN p.name as employee, c.name as company, r.since as start_year
""")
for record in results.get("records", []):
    print(f"{record['employee']} works at {record['company']} since {record['start_year']}")

In [ ]:
# Count and aggregate
results = neptune_store.execute_query("""
    MATCH (p:Person)
    RETURN count(p) as total, avg(p.age) as avg_age, max(p.age) as max_age
""")
stats = results.get("records", [{}])[0]
print(f"Total: {stats.get('total')}, Avg Age: {stats.get('avg_age'):.1f}")

## 步骤 5:图分析

### 获取邻居

遍历图以查找相连的节点。

In [ ]:
# Get immediate neighbors (depth=1)
neighbors = neptune_store.get_neighbors(
    node_id="alice",
    direction="both",
    depth=1
)
print(f"Alice's direct neighbors: {len(neighbors)}")

# Get neighbors up to 2 hops away
extended = neptune_store.get_neighbors(
    node_id="alice",
    direction="out",
    depth=2
)
print(f"Nodes within 2 hops: {len(extended)}")

### 最短路径

查找两个节点之间的最短路径。

In [ ]:
# Find shortest path
path = neptune_store.shortest_path(
    start_node_id="alice",
    end_node_id="charlie",
    max_depth=5
)

if path:
    print("Path found!")
    print(f"  Length: {path.get('length')}")
    print(f"  Nodes: {len(path.get('nodes', []))}")
    print(f"  Relationships: {len(path.get('relationships', []))}")
else:
    print("No path found between nodes")

## 步骤 6:图统计

获取图的全面统计信息。

In [ ]:
# Get graph statistics
stats = neptune_store.get_stats()

print("Graph Statistics:")
print(f"  Total nodes: {stats.get('node_count', 'N/A')}")
print(f"  Total relationships: {stats.get('relationship_count', 'N/A')}")

print("\nNode labels:")
for label, count in stats.get('label_counts', {}).items():
    print(f"  - {label}: {count}")

print("\nRelationship types:")
for rel_type, count in stats.get('relationship_type_counts', {}).items():
    print(f"  - {rel_type}: {count}")

## 步骤 7:连接管理

使用完毕后请务必关闭连接以释放资源。

In [ ]:
# Check connection status
status = neptune_store.get_status()
print(f"Connection status: {status}")

# Close the connection
neptune_store.close()
print("Connection closed")

## Neptune 特有的注意事项

### 原生元素 ID

Neptune 使用原生的 `~id` 来标识元素。在属性中包含 `id` 即可设置自定义 ID:

```python
# 创建带自定义 ID 的节点(在 properties 中包含 'id')
node = neptune_store.create_node(
    labels=["Person"],
    properties={"id": "my-custom-id", "name": "Test"}
)

# 创建带自动生成 UUID 的节点(在 properties 中省略 'id')
node = neptune_store.create_node(
    labels=["Person"],
    properties={"name": "Test"}
)

# 该 ID 会在内部的 id() 函数调用中使用:
# MATCH (n) WHERE id(n) = 'my-custom-id' RETURN n
```

### OpenCypher 注意事项

Amazon Neptune 数据库的 OpenCypher 实现与 Neo4j 有一些差异:

1. **没有 `shortestPath()` 函数**:请使用变长路径模式或 `allShortestPaths()`
2. **标签语法**:使用 `labels(n)` 函数获取节点标签
3. **属性更新**:使用 `SET n += {props}` 以实现合并行为

关于 Amazon Neptune 数据库所支持的完整 OpenCypher 规范,请参阅 [AWS 文档](https://docs.aws.amazon.com/neptune/latest/userguide/access-graph-opencypher.html)。

### Amazon Neptune Analytics

对于图算法、聚合和大规模遍历等分析型(OLAP)工作负载,可以考虑 [Amazon Neptune Analytics](https://docs.aws.amazon.com/neptune-analytics/latest/userguide/what-is-neptune-analytics.html)。Neptune Analytics 是对 Neptune 数据库的补充,为分析型查询提供优化的性能,而 Neptune 数据库则针对事务型(OLTP)工作负载进行了优化。

### 性能技巧

1. 创建多个节点/关系时**使用批量操作**
2. 在查询中**使用参数**以启用查询缓存
3. 使用 `LIMIT` 子句**限制结果集大小**

## 总结

本 Notebook 介绍了 Amazon Neptune 图存储的集成:

- **IAM 认证**:安全的 AWS SigV4 签名
- **CRUD 操作**:完整的节点和关系管理
- **OpenCypher 查询**:标准的图查询语言
- **图分析**:邻居与最短路径算法
- **统计与监控**:图指标和状态

### 关键要点

- Neptune 使用原生的 `~id` 来标识元素
- 生产环境推荐使用 IAM 认证
- Bolt 协议提供高效的二进制查询接口
- Semantica 抽象了 Neptune 特有的语法差异

### 下一步

- [图存储(Neo4j/FalkorDB)](09_Graph_Store.ipynb) - 与其他后端进行比较
- [构建知识图谱](07_Building_Knowledge_Graphs.ipynb) - 构建生产级知识图谱
- [图分析](10_Graph_Analytics.ipynb) - 高级分析算法